In [ ]:
import os, sys, json, asyncio, subprocess, textwrap, warnings
from datetime import datetime, timezone, timedelta

PROVIDER = "openai"
BACKEND  = "auto"
DB_PATH  = "/content/graphiti_tutorial.db"
GROUP_ID = "northwind_cs"


def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


if BACKEND == "auto":
    BACKEND = "falkordb_lite" if sys.version_info >= (3, 12) else "kuzu"

EXTRAS = {
    "falkordb_lite":   "graphiti-core[falkordblite]",
    "falkordb_remote": "graphiti-core[falkordb]",
    "kuzu":            "graphiti-core[kuzu]",
    "neo4j":           "graphiti-core",
}[BACKEND]

print(f"Installing {EXTRAS} (backend={BACKEND}, provider={PROVIDER}) ...")
pip(EXTRAS, "nest_asyncio")
if PROVIDER == "gemini":
    pip("graphiti-core[google-genai]")

os.environ["GRAPHITI_TELEMETRY_ENABLED"] = "false"
os.environ["SEMAPHORE_LIMIT"] = "8"
warnings.filterwarnings("ignore", category=DeprecationWarning)

import nest_asyncio
nest_asyncio.apply()


def run(coro):
    """Run an async coroutine from a notebook cell."""
    return asyncio.get_event_loop().run_until_complete(coro)


def rule(title):
    print("\n" + "=" * 78 + f"\n  {title}\n" + "=" * 78)

In [ ]:
from getpass import getpass

def get_key(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            return val
    except Exception:
        pass
    val = getpass(f"{name}: ")
    os.environ[name] = val
    return val


API_KEY = get_key("OPENAI_API_KEY" if PROVIDER == "openai" else "GOOGLE_API_KEY")

from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType, EntityNode
from graphiti_core.edges import EntityEdge
from graphiti_core.llm_client import LLMConfig
from graphiti_core.prompts.models import Message
from graphiti_core.search.search_filters import (
    SearchFilters, DateFilter, ComparisonOperator,
)
from graphiti_core.search.search_config_recipes import (
    NODE_HYBRID_SEARCH_RRF,
    NODE_HYBRID_SEARCH_MMR,
    EDGE_HYBRID_SEARCH_RRF,
    EDGE_HYBRID_SEARCH_NODE_DISTANCE,
    COMBINED_HYBRID_SEARCH_MMR,
    COMMUNITY_HYBRID_SEARCH_RRF,
)
from graphiti_core.utils.maintenance.graph_data_operations import clear_data
from pydantic import BaseModel, Field

if PROVIDER == "openai":
    from graphiti_core.llm_client import OpenAIClient
    from graphiti_core.embedder import OpenAIEmbedder, OpenAIEmbedderConfig
    from graphiti_core.cross_encoder.openai_reranker_client import OpenAIRerankerClient

    llm_config = LLMConfig(api_key=API_KEY, model=None, small_model=None, temperature=0.0)
    llm_client = OpenAIClient(config=llm_config)
    embedder = OpenAIEmbedder(config=OpenAIEmbedderConfig(
        api_key=API_KEY, embedding_model="text-embedding-3-small", embedding_dim=1024))
    cross_encoder = OpenAIRerankerClient(config=llm_config)
else:
    from graphiti_core.llm_client.gemini_client import GeminiClient
    from graphiti_core.embedder.gemini import GeminiEmbedder, GeminiEmbedderConfig
    from graphiti_core.cross_encoder.gemini_reranker_client import GeminiRerankerClient

    llm_config = LLMConfig(api_key=API_KEY, model=None, small_model=None, temperature=0.0)
    llm_client = GeminiClient(config=llm_config)
    embedder = GeminiEmbedder(config=GeminiEmbedderConfig(api_key=API_KEY, embedding_dim=1024))
    cross_encoder = GeminiRerankerClient(config=llm_config)

if BACKEND == "falkordb_lite":
    from redislite.async_falkordb_client import AsyncFalkorDB
    from graphiti_core.driver.falkordb_driver import FalkorDriver
    graph_driver = FalkorDriver(falkor_db=AsyncFalkorDB(dbfilename=DB_PATH), database="tutorial")
elif BACKEND == "falkordb_remote":
    from graphiti_core.driver.falkordb_driver import FalkorDriver
    graph_driver = FalkorDriver(host=os.getenv("FALKORDB_HOST", "localhost"),
                                port=int(os.getenv("FALKORDB_PORT", 6379)),
                                username=os.getenv("FALKORDB_USER") or None,
                                password=os.getenv("FALKORDB_PASSWORD") or None,
                                database="tutorial")
elif BACKEND == "kuzu":
    from graphiti_core.driver.kuzu_driver import KuzuDriver
    graph_driver = KuzuDriver(db=DB_PATH)
else:
    from graphiti_core.driver.neo4j_driver import Neo4jDriver
    graph_driver = Neo4jDriver(uri=get_key("NEO4J_URI"),
                               user=os.getenv("NEO4J_USER", "neo4j"),
                               password=get_key("NEO4J_PASSWORD"))

graphiti = Graphiti(
    graph_driver=graph_driver,
    llm_client=llm_client,
    embedder=embedder,
    cross_encoder=cross_encoder,
    store_raw_episode_content=True,
    max_coroutines=8,
)

run(graphiti.build_indices_and_constraints())
run(clear_data(graphiti.driver, group_ids=[GROUP_ID]))
print("Graphiti ready.")

In [ ]:
class Person(BaseModel):
    """A named human being: employee, customer contact, champion, or executive."""
    role: str | None = Field(None, description="Job title as written in the source, e.g. 'Head of Data Platform'.")
    email: str | None = Field(None, description="Email address if explicitly stated.")

class Company(BaseModel):
    """An organization: a customer account, prospect, vendor, or employer."""
    industry: str | None = Field(None, description="Industry sector if stated.")
    headcount: int | None = Field(None, description="Number of employees, if a number is given.")
    hq_city: str | None = Field(None, description="Headquarters city, if stated.")

class Product(BaseModel):
    """A software product, SKU, or service that a company can subscribe to."""
    category: str | None = Field(None, description="Product category, e.g. 'analytics platform'.")

class Incident(BaseModel):
    """A discrete operational failure: outage, degradation, data loss, or breach."""
    severity: str | None = Field(None, description="Severity label if stated, e.g. 'SEV-1'.")
    duration_minutes: int | None = Field(None, description="Duration in minutes if stated.")

ENTITY_TYPES = {
    "Person": Person,
    "Company": Company,
    "Product": Product,
    "Incident": Incident,
}


class WORKS_AT(BaseModel):
    """A person is currently employed by a company."""
    role: str | None = Field(None, description="Their title at that company.")
    started_at: str | None = Field(None, description="ISO date the employment started, if stated.")

class SUBSCRIBES_TO(BaseModel):
    """A company pays for a product under a specific plan."""
    plan: str | None = Field(None, description="Plan tier name, e.g. 'Team', 'Enterprise'.")
    seats: int | None = Field(None, description="Number of licensed seats.")
    mrr_usd: float | None = Field(None, description="Monthly recurring revenue in USD.")

class AFFECTED_BY(BaseModel):
    """A company or product was impacted by an incident."""
    impact: str | None = Field(None, description="Short description of the impact.")

EDGE_TYPES = {
    "WORKS_AT": WORKS_AT,
    "SUBSCRIBES_TO": SUBSCRIBES_TO,
    "AFFECTED_BY": AFFECTED_BY,
}

EDGE_TYPE_MAP = {
    ("Person", "Company"):   ["WORKS_AT"],
    ("Company", "Product"):  ["SUBSCRIBES_TO"],
    ("Company", "Incident"): ["AFFECTED_BY"],
    ("Product", "Incident"): ["AFFECTED_BY"],
    ("Entity", "Entity"):    [],
}


def T(s):
    return datetime.fromisoformat(s).replace(tzinfo=timezone.utc)

EPISODES = [
    dict(name="crm-note-001", source=EpisodeType.text, reference_time=T("2024-11-05"),
         source_description="CRM account note",
         episode_body=(
             "Priya Raman joined Northwind Analytics as Head of Data Platform. "
             "Northwind Analytics is a 400-person logistics analytics company "
             "headquartered in Bengaluru.")),

    dict(name="billing-event-001", source=EpisodeType.json, reference_time=T("2025-01-12"),
         source_description="Billing system export",
         episode_body=json.dumps({
             "account": "Northwind Analytics",
             "product": "Graphiti Cloud",
             "plan": "Team",
             "seats": 25,
             "mrr_usd": 1250,
             "event": "subscription_started",
         })),

    dict(name="support-chat-001", source=EpisodeType.message, reference_time=T("2025-02-03"),
         source_description="Support chat transcript",
         episode_body=(
             "Priya Raman: We keep bumping into the 25-seat ceiling on the Team plan.\n"
             "Support Agent: Enterprise removes the seat cap and adds SSO. "
             "I'll loop in your account executive, Daniel Osei.")),

    dict(name="crm-note-002", source=EpisodeType.text, reference_time=T("2025-03-18"),
         source_description="CRM account note",
         episode_body=(
             "Northwind Analytics upgraded from the Team plan to the Enterprise plan "
             "for Graphiti Cloud, expanding to 120 seats at $9,600 MRR.")),

    dict(name="incident-report-001", source=EpisodeType.text, reference_time=T("2025-04-02"),
         source_description="Postmortem document",
         episode_body=(
             "Graphiti Cloud suffered a SEV-1 outage in ap-south-1 lasting 90 minutes. "
             "Northwind Analytics was affected: their nightly ingestion pipeline failed. "
             "Priya Raman escalated to the on-call team.")),

    dict(name="crm-note-003", source=EpisodeType.text, reference_time=T("2025-06-20"),
         source_description="CRM account note",
         episode_body=(
             "Priya Raman has left Northwind Analytics. She is now VP of Engineering "
             "at Fleetsignal, a fleet telematics startup in Pune.")),

    dict(name="support-chat-002", source=EpisodeType.message, reference_time=T("2025-08-15"),
         source_description="Support chat transcript",
         episode_body=(
             "Arjun Mehta: Hi, I'm the new Director of Analytics at Northwind Analytics, "
             "taking over Graphiti Cloud from Priya.\n"
             "Support Agent: Welcome Arjun — you're now the primary contact on the "
             "Enterprise plan.")),
]

rule("3. INGESTING EPISODES")
for ep in EPISODES:
    res = run(graphiti.add_episode(
        group_id=GROUP_ID,
        entity_types=ENTITY_TYPES,
        edge_types=EDGE_TYPES,
        edge_type_map=EDGE_TYPE_MAP,
        custom_extraction_instructions=(
            "Treat plan names, seat counts and MRR as attributes of the "
            "SUBSCRIBES_TO fact, not as separate entities."
        ),
        **ep,
    ))
    nodes = ", ".join(f"{n.name} [{'/'.join(l for l in n.labels if l != 'Entity')}]" for n in res.nodes)
    print(f"\n{ep['name']} @ {ep['reference_time'].date()}")
    print(f"  nodes: {nodes or '—'}")
    for e in res.edges:
        print(f"  fact : ({e.name}) {e.fact}")

In [ ]:
_EDGE_PATTERN = ("MATCH (n:Entity)-[:RELATES_TO]->(e:RelatesToNode_)-[:RELATES_TO]->(m:Entity)"
                 if BACKEND == "kuzu" else
                 "MATCH (n:Entity)-[e:RELATES_TO]->(m:Entity)")

CYPHER_ALL_FACTS = _EDGE_PATTERN + """
WHERE e.group_id = $group_id
RETURN n.name AS src, m.name AS dst, e.name AS type, e.fact AS fact,
       e.valid_at AS valid_at, e.invalid_at AS invalid_at, e.expired_at AS expired_at
"""

def show_facts(title="ALL FACTS"):
    records, _, _ = run(graphiti.driver.execute_query(CYPHER_ALL_FACTS, group_id=GROUP_ID))
    rule(title)
    for r in records:
        state = "EXPIRED" if r.get("expired_at") else "CURRENT"
        print(f"[{state:7}] ({r['type']}) {r['fact']}")
        print(f"            valid_at={str(r['valid_at'])[:10] or '—'}  "
              f"invalid_at={str(r['invalid_at'])[:10] if r['invalid_at'] else '—'}")

show_facts("4. FACT LEDGER (note the expired employment + plan facts)")


rule("5. HYBRID FACT SEARCH")
for q in ["Who is the main contact at Northwind Analytics?",
          "What plan is Northwind on?"]:
    print(f"\nQ: {q}")
    for e in run(graphiti.search(q, group_ids=[GROUP_ID], num_results=5)):
        marker = "×" if e.invalid_at else "✓"
        print(f"  {marker} {e.fact}")


rule("6. SEARCH RECIPES")

node_res = run(graphiti.search_("logistics analytics customer",
                                config=NODE_HYBRID_SEARCH_RRF, group_ids=[GROUP_ID]))
print("\nEntities (RRF):")
for n in node_res.nodes[:5]:
    labels = "/".join(l for l in n.labels if l != "Entity")
    print(f"  • {n.name} [{labels}] — {(n.summary or '')[:90]}")
    if n.attributes:
        print(f"      attrs: { {k: v for k, v in n.attributes.items() if v is not None} }")

center = next((n for n in node_res.nodes if "Northwind" in n.name), None)
if center:
    cfg = EDGE_HYBRID_SEARCH_NODE_DISTANCE.model_copy(deep=True)
    cfg.limit = 10
    around = run(graphiti.search_("account status subscription contacts incidents",
                                  config=cfg, group_ids=[GROUP_ID],
                                  center_node_uuid=center.uuid))
    print(f"\nFacts ranked by graph distance from '{center.name}':")
    for e in around.edges[:8]:
        print(f"  • {e.fact}")

combined = run(graphiti.search_("Northwind Analytics history",
                                config=COMBINED_HYBRID_SEARCH_MMR, group_ids=[GROUP_ID]))
print(f"\nCombined MMR: {len(combined.edges)} edges, {len(combined.nodes)} nodes, "
      f"{len(combined.episodes)} episodes, {len(combined.communities)} communities")

custom = EDGE_HYBRID_SEARCH_RRF.model_copy(deep=True)
custom.limit = 15
custom.edge_config.sim_min_score = 0.7
custom.edge_config.bfs_max_depth = 2

In [ ]:
rule("7. FILTERED + POINT-IN-TIME SEARCH")

only_subs = SearchFilters(edge_types=["SUBSCRIBES_TO"])
print("\nSUBSCRIBES_TO facts only:")
for e in run(graphiti.search("plan and seats", group_ids=[GROUP_ID],
                             search_filter=only_subs, num_results=10)):
    print(f"  • {e.fact}  (valid {str(e.valid_at)[:10]} → {str(e.invalid_at)[:10] if e.invalid_at else 'now'})")

people = run(graphiti.search_("who is involved with this account",
                             config=NODE_HYBRID_SEARCH_MMR, group_ids=[GROUP_ID],
                             search_filter=SearchFilters(node_labels=["Person"])))
print("\nPerson nodes only:", ", ".join(n.name for n in people.nodes))

def as_of(t: datetime) -> SearchFilters:
    return SearchFilters(
        valid_at=[[DateFilter(date=t, comparison_operator=ComparisonOperator.less_than_equal)]],
        invalid_at=[
            [DateFilter(comparison_operator=ComparisonOperator.is_null)],
            [DateFilter(date=t, comparison_operator=ComparisonOperator.greater_than)],
        ],
    )

for t in [T("2025-02-01"), T("2025-05-01"), T("2025-09-01")]:
    print(f"\nWorld state as of {t.date()} — 'Northwind plan and staffing':")
    hits = run(graphiti.search("Northwind Analytics plan, seats, and who works there",
                               group_ids=[GROUP_ID], search_filter=as_of(t), num_results=6))
    for e in hits:
        print(f"  • {e.fact}")


rule("8. COMMUNITY DETECTION")
try:
    communities, _edges = run(graphiti.build_communities(group_ids=[GROUP_ID]))
    for c in communities:
        print(f"\n▣ {c.name}\n  {textwrap.fill(c.summary or '', 74, subsequent_indent='  ')}")

    cres = run(graphiti.search_("overall account themes",
                                config=COMMUNITY_HYBRID_SEARCH_RRF, group_ids=[GROUP_ID]))
    print(f"\nCommunity search hits: {[c.name for c in cres.communities]}")
except Exception as ex:
    print(f"Community building skipped ({type(ex).__name__}: {ex})")


rule("9. DIRECT GRAPH OPERATIONS")

src = EntityNode(name="Fleetsignal", group_id=GROUP_ID, labels=["Entity", "Company"],
                 summary="Fleet telematics startup based in Pune.")
dst = EntityNode(name="Graphiti Cloud", group_id=GROUP_ID, labels=["Entity", "Product"],
                 summary="Temporal knowledge graph platform.")
edge = EntityEdge(source_node_uuid=src.uuid, target_node_uuid=dst.uuid, group_id=GROUP_ID,
                  name="EVALUATING", fact="Fleetsignal is evaluating Graphiti Cloud.",
                  created_at=datetime.now(timezone.utc), valid_at=T("2025-09-01"), episodes=[])
run(graphiti.add_triplet(src, edge, dst))
print("Manual triplet written.")

recent = run(graphiti.retrieve_episodes(reference_time=datetime.now(timezone.utc),
                                        last_n=3, group_ids=[GROUP_ID]))
print("\nMost recent episodes:")
for ep in recent:
    print(f"  • {ep.name} ({ep.source.value}, {ep.valid_at.date()}): {ep.content[:70]}...")

In [ ]:
rule("10. GROUNDED Q&A OVER THE GRAPH")

class Answer(BaseModel):
    answer: str = Field(description="Answer in 2-3 sentences, grounded only in the provided facts.")
    supporting_facts: list[str] = Field(description="Verbatim facts used, from the context.")
    confidence: float = Field(description="0.0-1.0.")


def build_context(query: str, at_time: datetime | None = None, k: int = 12) -> str:
    cfg = COMBINED_HYBRID_SEARCH_MMR.model_copy(deep=True)
    cfg.limit = k
    res = run(graphiti.search_(query, config=cfg, group_ids=[GROUP_ID],
                               search_filter=as_of(at_time) if at_time else None))
    lines = ["# FACTS (with validity windows)"]
    for e in res.edges:
        window = f"valid from {str(e.valid_at)[:10] if e.valid_at else '?'}"
        window += f" until {str(e.invalid_at)[:10]}" if e.invalid_at else " (still current)"
        lines.append(f"- {e.fact}  [{window}]")
    lines.append("\n# ENTITIES")
    for n in res.nodes:
        lines.append(f"- {n.name}: {(n.summary or '')[:160]}")
    return "\n".join(lines)


def ask(query: str, at_time: datetime | None = None) -> dict:
    ctx = build_context(query, at_time)
    stamp = f"\nToday's date for the purposes of this question: {at_time.date()}" if at_time else ""
    prompt = (
        "You are an account-intelligence assistant with access to a temporal knowledge graph.\n"
        "Use ONLY the facts below. Respect validity windows: never state an expired fact as "
        "currently true — say what changed and when. If the facts don't cover it, say so."
        f"{stamp}\n\n{ctx}\n\nQuestion: {query}"
    )
    return run(graphiti.llm_client.generate_response(
        messages=[Message(role="user", content=prompt)], response_model=Answer))


for q, t in [("Who runs data platform at Northwind Analytics, and has that changed?", None),
             ("Is Northwind a reliable account? Mention any incidents.", None),
             ("Who was our champion at Northwind and what were they paying?", T("2025-02-15"))]:
    out = ask(q, t)
    print(f"\nQ: {q}" + (f"   (asked as of {t.date()})" if t else ""))
    print(textwrap.fill(f"A: {out['answer']}", 78))
    print(f"   confidence={out.get('confidence')}")


rule("11. GRAPH VISUALIZATION")
try:
    import networkx as nx, matplotlib.pyplot as plt

    records, _, _ = run(graphiti.driver.execute_query(CYPHER_ALL_FACTS, group_id=GROUP_ID))
    G = nx.DiGraph()
    for r in records:
        G.add_edge(r["src"], r["dst"], label=r["type"] or "RELATES_TO",
                   expired=bool(r.get("expired_at")))
    pos = nx.spring_layout(G, seed=7, k=1.4)
    plt.figure(figsize=(13, 8))
    live = [(u, v) for u, v, d in G.edges(data=True) if not d["expired"]]
    dead = [(u, v) for u, v, d in G.edges(data=True) if d["expired"]]
    nx.draw_networkx_nodes(G, pos, node_size=2400, node_color="#e8eef7", edgecolors="#33475b")
    nx.draw_networkx_edges(G, pos, edgelist=live, edge_color="#2d6cdf", width=2, arrowsize=18)
    nx.draw_networkx_edges(G, pos, edgelist=dead, edge_color="#c44", style="dashed",
                           width=1.4, arrowsize=14)
    nx.draw_networkx_labels(G, pos, font_size=8)
    nx.draw_networkx_edge_labels(G, pos, font_size=6,
                                 edge_labels={(u, v): d["label"] for u, v, d in G.edges(data=True)})
    plt.title("Northwind knowledge graph — solid = current fact, dashed red = expired")
    plt.axis("off"); plt.tight_layout(); plt.show()
except Exception as ex:
    print(f"Visualization skipped: {ex}")

rule("TOKEN USAGE BY PROMPT")
try:
    usage = graphiti.token_tracker.get_usage()
    total = 0
    for name, u in sorted(usage.items(), key=lambda kv: -kv[1].total_tokens):
        total += u.total_tokens
        print(f"  {name:38} calls={u.call_count:3}  tokens={u.total_tokens:,}")
    print(f"  {'TOTAL':38}            tokens={total:,}")
except Exception as ex:
    print(f"(token tracking unavailable: {ex})")

run(graphiti.close())
print("\nDone. The graph file persists at", DB_PATH)

Installing graphiti-core[falkordblite] (backend=falkordb_lite, provider=openai) ...
